# Session 2: the alias formula and the Nyquist limit
**A. Real-world problem.** Session 1 found a sawtooth by looking at pictures. Engineers meet
the same sawtooth in sound cards, oscilloscopes and cameras, and they need a formula and a
rule: how fast must we sample so that nothing is mistaken for something else?

**Plan.** (1) sample a sine wave, (2) show that two different frequencies give *identical*
samples, (3) write the alias formula, (4) use it to design a frame rate.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: sampling a sine wave
A signal x(t) = sin(2 pi f t) is measured only at the instants t_n = n / fs (n = 0, 1, 2, ...),
where fs is the sampling rate (samples per second, hertz). The numbers we keep are

    x[n] = sin(2 pi f n / fs)

In [ ]:
def sample_sine(f, fs, n_samples, phase=0.0):
    """The samples x[n] = sin(2 pi f n / fs + phase), n = 0 .. n_samples - 1."""
    n = np.arange(n_samples)
    return np.sin(2.0 * np.pi * f * n / fs + phase)

## Step 2: two frequencies, one set of samples
Add a whole number k of sampling rates to the frequency: f' = f + k fs. Then
2 pi f' n / fs = 2 pi f n / fs + 2 pi k n, and adding a whole number of turns (2 pi k n) to an
angle changes nothing. So f and f + k fs give exactly the same samples. Also sin(-a) = -sin(a),
so f and -f give the same samples up to a sign. After sampling, no maths can tell them apart.

In [ ]:
def same_samples(f1, f2, fs, n_samples=64, phase=0.0, tol=1e-9):
    return bool(np.allclose(sample_sine(f1, fs, n_samples, phase), sample_sine(f2, fs, n_samples, phase), atol=tol))

## Step 3: the alias frequency
Choose the whole number k that brings f closest to a multiple of fs (k = round(f / fs)). What
is left over is the *signed alias* f - k fs, always between -fs/2 and +fs/2. Its size is the
frequency the samples seem to have:

    f_alias = | f - k fs |,   k = round(f / fs),   so 0 <= f_alias <= fs / 2

The limit fs / 2 is the **Nyquist frequency**. Below it nothing is confused; above it we get an alias.

In [ ]:
def nearest_multiple(f, fs):
    """k = round(f / fs), rounding halves upward (a tie only happens at f = fs/2 + j fs)."""
    return int(np.floor(f / fs + 0.5))


def signed_alias(f, fs):
    return f - fs * nearest_multiple(f, fs)


def alias_frequency(f, fs):
    return abs(signed_alias(f, fs))


def nyquist(fs):
    return fs / 2.0

## Step 4: design a frame rate
To keep a signal whose highest frequency is f_max, the sampling rate must satisfy
fs > 2 f_max. A wheel with N identical spokes looks the same after 1/N of a turn, so its
picture repeats N f times per second: the signal frequency is N f, not f. `pattern_apparent_rev`
gives the apparent turning speed of such a wheel in rev/s.

In [ ]:
def min_sampling_rate(f_max):
    """Smallest rate that must be exceeded (strictly) to avoid aliasing: 2 f_max."""
    return 2.0 * f_max


def pattern_apparent_rev(f_rev, n_spokes, fs):
    """Apparent signed speed (rev/s) of a wheel with n identical spokes turning f_rev rev/s."""
    return signed_alias(n_spokes * f_rev, fs) / n_spokes

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    f, fs = 40.0, 50.0
    t = np.linspace(0, 0.2, 2000)
    tn = np.arange(0, 0.2, 1 / fs)
    fa = signed_alias(f, fs)
    print(f"f = {f} Hz at fs = {fs} Hz: k = {nearest_multiple(f, fs)}, signed alias = {fa}, alias = {alias_frequency(f, fs)} Hz")
    fig, ax = plt.subplots(figsize=(7, 3.6))
    ax.plot(t, np.sin(2 * np.pi * f * t), color="lightgray", label="true 40 Hz signal")
    ax.plot(t, np.sin(2 * np.pi * fa * t), color="tab:blue", label="alias (-10 Hz) through the samples")
    ax.plot(tn, np.sin(2 * np.pi * f * tn), "ko", label="samples at 50 Hz")
    ax.set_xlabel("time t (s)")
    ax.set_ylabel("x(t) (arbitrary units)")
    ax.set_title("Samples of a 40 Hz sine at 50 Hz fit a slow 10 Hz sine")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper right", fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "samples_alias.png", dpi=120)

    f_axis = np.linspace(0, 150, 1501)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(f_axis, [alias_frequency(x, fs) for x in f_axis], color="tab:blue", label="f_alias")
    ax2.axhline(nyquist(fs), color="tab:red", ls="--", label="Nyquist fs/2 = 25 Hz")
    ax2.set_xlabel("true frequency f (Hz)")
    ax2.set_ylabel("alias frequency (Hz)")
    ax2.set_title("Folding at fs = 50 Hz: zigzag between 0 and fs/2")
    ax2.grid(alpha=0.3)
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "folding.png", dpi=120)
    print("4-blade fan at 12 rev/s filmed at 30 fps looks like", pattern_apparent_rev(12, 4, 30), "rev/s")
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Print `alias_frequency(f, 50)` for f = 5, 25, 30, 45, 55 and 80 Hz. Predict each value first and mark which are aliased.
2. **Stretch.** Write `frames_needed(n_blades, f_rev)` that returns the smallest whole frame rate that is strictly above the Nyquist limit for a fan, and test it with 4 blades at 12 rev/s (expect 97).
3. **Explain.** Explain why 120 fps shows the 4-blade fan correctly but 60 fps does not, even though 60 is already above the fan speed 12 rev/s.

## Assessment
- **A. Conceptual.** After a signal has been sampled, can any processing of the samples tell a 40 Hz sine from its 10 Hz alias at fs = 50 Hz? Why is the only real cure a change before sampling?
- **B. Numerical.** A 130 Hz signal is sampled at 100 Hz. Find the alias frequency and give two other frequencies that give the same samples.
- **C. Graph and simulation interpretation.** On the folding graph for fs = 50 Hz, at which true frequencies from 0 to 150 Hz is the alias exactly 10 Hz?
- **D. Design and optimisation challenge.** Design: a four-blade fan turns 12 rev/s. A phone can film at 30, 60, 100 or 120 frames per second. Which rates show the true speed and direction? What is the minimum, and what does a higher rate cost?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("worked example: 40 Hz sampled at 50 Hz has alias 10 Hz (k = 1)",
                   L.nearest_multiple(40, 50) == 1 and abs(L.alias_frequency(40, 50) - 10.0) < 1e-9
                   and abs(L.signed_alias(40, 50) + 10.0) < 1e-9))
    s40 = L.sample_sine(40, 50, 50)
    s10 = L.sample_sine(-10, 50, 50)
    checks.append(("worked example: the samples of 40 Hz equal the samples of -10 Hz", np.allclose(s40, s10, atol=1e-9)))
    checks.append(("hand calculation: x[1] = sin(2 pi 0.8) = -0.951", abs(s40[1] + 0.951057) < 1e-5))
    checks.append(("f and f + k fs give identical samples", L.same_samples(7.0, 7.0 + 3 * 50.0, 50.0)))
    checks.append(("f and fs - f differ only by a sign of the samples",
                   np.allclose(L.sample_sine(40, 50, 40), -L.sample_sine(10, 50, 40), atol=1e-9)))
    checks.append(("a signal below Nyquist is not changed (12 Hz at 50 Hz)", abs(L.alias_frequency(12, 50) - 12.0) < 1e-9))
    checks.append(("exactly Nyquist: 25 Hz at 50 Hz gives alias 25 Hz", abs(L.alias_frequency(25, 50) - 25.0) < 1e-9))
    checks.append(("exactly fs gives alias 0 (DC): 50 Hz at 50 Hz", abs(L.alias_frequency(50, 50)) < 1e-9))
    fa = [L.alias_frequency(x, 50) for x in np.linspace(0, 500, 2001)]
    checks.append(("alias frequency always lies between 0 and fs/2", 0 <= min(fa) and max(fa) <= 25 + 1e-9))
    checks.append(("design: the minimum rate for a 48 Hz pattern is 96 Hz", abs(L.min_sampling_rate(48) - 96.0) < 1e-9))
    checks.append(("worked example (design): a 4-blade fan at 12 rev/s filmed at 30 fps looks like -3 rev/s",
                   abs(L.pattern_apparent_rev(12, 4, 30) + 3.0) < 1e-9))
    checks.append(("the same fan filmed at 120 fps is seen correctly (12 rev/s)", abs(L.pattern_apparent_rev(12, 4, 120) - 12.0) < 1e-9))
    checks.append(("one-spoke wheel agrees with Session 1: 28 rev/s at 30 fps is -2 rev/s", abs(L.pattern_apparent_rev(28, 1, 30) + 2.0) < 1e-9))
    checks.append(("hand calculation: 130 Hz sampled at 100 Hz has alias 30 Hz; 70 Hz and 170 Hz alias to 30 Hz too",
                   all(abs(L.alias_frequency(x, 100) - 30.0) < 1e-9 for x in (130, 70, 170))))
    zig = [x for x in np.arange(0, 150, 0.5) if abs(L.alias_frequency(x, 50) - 10.0) < 1e-9]
    checks.append(("graph reading: at fs = 50 the alias is 10 Hz for f = 10, 40, 60, 90, 110, 140", zig == [10.0, 40.0, 60.0, 90.0, 110.0, 140.0]))
    checks.append(("design: the 4-blade 12 rev/s fan looks -3 rev/s at 60 fps but +12 rev/s (correct) at 100 and 120 fps",
                   abs(L.pattern_apparent_rev(12, 4, 60) + 3.0) < 1e-9 and abs(L.pattern_apparent_rev(12, 4, 100) - 12.0) < 1e-9
                   and abs(L.pattern_apparent_rev(12, 4, 120) - 12.0) < 1e-9))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
